In [1]:
import pandas as pd
import numpy as np

In [2]:
data = {
    'date': [
        '2021-12-01', '01-12-2022', '2022/12/01', '12-01-2021', 
        '2023.01.15', 'Jan 5, 2023', '2022-12-01', 'not_a_date'
    ],
    'country': [
        'USA', 'U.S.A.', 'America', 'United States', 
        'pakistan', 'PK', 'Pak.', '  United States  '
    ],
    'name': [
        'Aammar', 'Amaar', 'Hamza', 'Hazma', 
        'aammar', 'AAMMAR', 'Dr. Hamza', 'Hamza '
    ],
    'CNIC': [
        '42101-1234567-1', '4210112345671', '42101 1234567 1', '35202-98765432', 
        '12345-6789012-3', '42101-1234567', 'N/A', '42101-ABCDEFG-1'
    ],
    'sales_2020': [100, 200, None, 200, 50, '100', 300, 0],
    'sales_2021': [None, 150, 300, 150, 200, 250, 'missing', -50]
}
df = pd.DataFrame(data)

In [3]:
df['cleaned_date'] = pd.to_datetime(df['date'], errors='coerce', format='mixed')

df['formatted_date'] = df['cleaned_date'].dt.strftime('%d-%m-%Y')
df.drop(columns=['cleaned_date'], inplace=True)
df.head(10)
df['date']=df['formatted_date']
df.drop(columns=['formatted_date'], inplace=True)

In [4]:
# Harmonize the name of the coutry
country_mapping = {'USA': 'United States', 'U.S.A.': 'United States', 'America': 'United States', 'PK': 'Pakistan', 'pakistan':'Pakistan', 'Pak.':'Pakistan'}
df['country'] = df['country'].replace(country_mapping)
df.head(10)

,date,country,name,CNIC,sales_2020,sales_2021
0,01-12-2021,United States,Aammar,42101-1234567-1,100,None
1,12-01-2022,United States,Amaar,4210112345671,200,150
2,01-12-2022,United States,Hamza,42101 1234567 1,None,300
3,01-12-2021,United States,Hazma,35202-98765432,200,150
4,15-01-2023,Pakistan,aammar,12345-6789012-3,50,200
5,05-01-2023,Pakistan,AAMMAR,42101-1234567,100,250
6,01-12-2022,Pakistan,Dr. Hamza,N/A,300,missing
7,NaN,United States,Hamza,42101-ABCDEFG-1,0,-50


In [5]:
# Correct the typographical Mistakes in name
df['name'] = df['name'].replace({'Amaar': 'Aammar','AAMMAR': 'Aammar', 'Hazma': 'Hamza', 'Dr. Hazma': 'Hamza'})
df.head(10)

,date,country,name,CNIC,sales_2020,sales_2021
0,01-12-2021,United States,Aammar,42101-1234567-1,100,None
1,12-01-2022,United States,Aammar,4210112345671,200,150
2,01-12-2022,United States,Hamza,42101 1234567 1,None,300
3,01-12-2021,United States,Hamza,35202-98765432,200,150
4,15-01-2023,Pakistan,aammar,12345-6789012-3,50,200
5,05-01-2023,Pakistan,Aammar,42101-1234567,100,250
6,01-12-2022,Pakistan,Dr. Hamza,N/A,300,missing
7,NaN,United States,Hamza,42101-ABCDEFG-1,0,-50


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        7 non-null      str   
 1   country     8 non-null      str   
 2   name        8 non-null      str   
 3   CNIC        8 non-null      str   
 4   sales_2020  7 non-null      object
 5   sales_2021  7 non-null      object
dtypes: object(2), str(4)
memory usage: 516.0+ bytes


In [7]:
# convert sales_2020 into float
df['sales_2020']=df['sales_2020'].astype(float)

In [8]:
df['sales_2021'] = pd.to_numeric(df['sales_2021'], errors='coerce')
df['sales_2021'] = df['sales_2021'].fillna(0)
df['sales_2021'] = df['sales_2021'].clip(lower=0)

In [9]:
df['clean_cnic'] = df['CNIC'].str.replace(r'\D', '', regex=True)

def format_cnic(val):
    if len(val) == 13:
        return f"{val[:5]}-{val[5:12]}-{val[12]}"
    return val

df['formatted_cnic'] = df['clean_cnic'].apply(format_cnic)

print(df[['CNIC', 'formatted_cnic']])
df.drop(columns=['clean_cnic'], inplace=True)

              CNIC   formatted_cnic
0  42101-1234567-1  42101-1234567-1
1    4210112345671  42101-1234567-1
2  42101 1234567 1  42101-1234567-1
3   35202-98765432  35202-9876543-2
4  12345-6789012-3  12345-6789012-3
5    42101-1234567     421011234567
6              N/A                 
7  42101-ABCDEFG-1           421011


In [10]:
df = df.drop(df[df['sales_2021'] <= df['sales_2020']].index)
df.head()

,date,country,name,CNIC,sales_2020,sales_2021,formatted_cnic
2,01-12-2022,United States,Hamza,42101 1234567 1,NaN,300.0,42101-1234567-1
4,15-01-2023,Pakistan,aammar,12345-6789012-3,50.0,200.0,12345-6789012-3
5,05-01-2023,Pakistan,Aammar,42101-1234567,100.0,250.0,421011234567
